# Phase 7 & 8 — Data Splitting & Preprocessing Pipeline

We take our strict 100-image subset and deterministically split it (70/15/15) for training, validation, and testing. Then, we define our preprocessing transforms (both strict and lightly augmented).

In [1]:
# ── Step 0: Imports ───────────────────────────────────────────────────────────
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
import torchvision.transforms as transforms
import torch

MANIFEST_PATH = '../data/processed/subset_manifest.csv'
SPLIT_MANIFEST_PATH = '../data/processed/split_manifest.csv'
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

In [2]:
# ── Step 1: Phase 7 - Data Splitting (70/15/15) ───────────────────────────────
print('Loading manifest...')
manifest = pd.read_csv(MANIFEST_PATH)

# Perform stratified split
train_df, temp_df = train_test_split(manifest, test_size=0.30, random_state=SEED, stratify=manifest['label'])
val_df, test_df = train_test_split(temp_df, test_size=0.50, random_state=SEED, stratify=temp_df['label'])

# Update 'split' column
manifest.loc[train_df.index, 'split'] = 'train'
manifest.loc[val_df.index, 'split'] = 'val'
manifest.loc[test_df.index, 'split'] = 'test'

print('\n--- Split Results ---')
print(f"Train : {len(train_df)} samples (Class 0: {(train_df['label'] == 0).sum()}, Class 1: {(train_df['label'] == 1).sum()})")
print(f"Val   : {len(val_df)} samples (Class 0: {(val_df['label'] == 0).sum()}, Class 1: {(val_df['label'] == 1).sum()})")
print(f"Test  : {len(test_df)} samples (Class 0: {(test_df['label'] == 0).sum()}, Class 1: {(test_df['label'] == 1).sum()})")

# Save split manifest
manifest.to_csv(SPLIT_MANIFEST_PATH, index=False)
print(f'\n✅ Split manifest saved to {SPLIT_MANIFEST_PATH}')

Loading manifest...

--- Split Results ---
Train : 70 samples (Class 0: 35, Class 1: 35)
Val   : 15 samples (Class 0: 7, Class 1: 8)
Test  : 15 samples (Class 0: 8, Class 1: 7)

✅ Split manifest saved to ../data/processed/split_manifest.csv


In [3]:
# ── Step 2: Phase 8 - Preprocessing Pipeline ──────────────────────────────────
print('Defining Preprocessing Transforms...')

# Note: Medical images shouldn't have aggressive augmentations that alter anatomy.
# Normalization uses standard MedMNIST mean/std.

transform_eval = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])
])

transform_train = transforms.Compose([
    transforms.ToTensor(),
    transforms.RandomRotation(degrees=10), # Light rotation
    transforms.Normalize(mean=[0.5], std=[0.5])
])

print('\nEvaluation Transforms (Strict):')
print(transform_eval)
print('\nTraining Transforms (Light Augmentation):')
print(transform_train)

print('\n✅ Preprocessing Pipeline Defined.')

Defining Preprocessing Transforms...

Evaluation Transforms (Strict):
Compose(
    ToTensor()
    Normalize(mean=[0.5], std=[0.5])
)

Training Transforms (Light Augmentation):
Compose(
    ToTensor()
    RandomRotation(degrees=[-10.0, 10.0], interpolation=nearest, expand=False, fill=0)
    Normalize(mean=[0.5], std=[0.5])
)

✅ Preprocessing Pipeline Defined.
